In [1]:
!nvidia-smi

Sat Aug 29 11:28:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!apt-get update -qq
!apt-get install -y zstd pciutils
!pip -q install pandas openpyxl requests

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids
The following NEW packages will be installed:
  libpci3 pci.ids pciutils zstd
0 upgraded, 4 newly installed, 0 to remove and 77 not upgraded.
Need to get 946 kB of archives.
After this operation, 3,276 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 946 

In [4]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

response = requests.get(
    "http://127.0.0.1:11434/api/tags"
)

print("Ollama status:", response.status_code)

Ollama status: 200


In [7]:
!ollama pull gemma3:4b

In [8]:
from pathlib import Path
import pandas as pd

INPUT_FILE = Path(
    "/content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_LLM_Benchmark.xlsx"
)

RESULT_FILE = Path(
    "/content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_Gemma3_Results.xlsx"
)

print("Input exists:", INPUT_FILE.exists())

df = pd.read_excel(
    INPUT_FILE,
    sheet_name="Benchmark"
)

print("Number of records:", len(df))

Input exists: True
Number of records: 500


In [9]:
df["gemma_output"] = ""
df["gemma_raw"] = ""
df["gemma_status"] = ""
df["gemma_runtime_sec"] = ""

df.to_excel(
    RESULT_FILE,
    index=False
)

print("Created:")
print(RESULT_FILE)

Created:
/content/drive/MyDrive/LLMValidation_LabeledIn/LabeledIn_Gemma3_Results.xlsx


In [11]:
import json
import time
import requests

MODEL = "gemma3:4b"

URL = "http://127.0.0.1:11434/api/chat"


SYSTEM_MESSAGE = """
You are a biomedical information extraction system.
Follow the user's extraction instructions exactly.
Return only the requested JSON output.
""".strip()


EXTRACTION_TEMPLATE = """
Extract ALL diseases, disorders, syndromes, symptoms, or clinical conditions
for which the drug is explicitly indicated in the following
Indications and Usage text.

Rules:

1. Read the entire supplied text.
2. Include every explicitly stated clinical indication.
3. Include symptoms or clinical conditions when explicitly indicated
   for treatment, prevention, management, or relief.
4. Preserve clinically meaningful disease subtypes and modifiers.
5. Use wording from the supplied text as closely as possible.
6. Do not intentionally normalize or generalize disease names.
7. Do not infer conditions that are not explicitly supported by the text.
8. Exclude conditions explicitly stated as NOT indicated or NOT recommended.
9. Exclude causative organisms when they merely describe the cause of infection.
10. Exclude mechanisms, procedures, dosing information,
    treatment regimens, and patient demographics.
11. Remove duplicate clinical conditions.

Return ONLY valid JSON:

{{"diseases":["condition 1","condition 2"]}}

If there are genuinely no indications:

{{"diseases":[]}}

INDICATIONS AND USAGE:

{indication_text}
""".strip()

In [12]:
def parse_output(raw_output):

    raw_output = str(raw_output).strip()

    try:

        parsed = json.loads(
            raw_output
        )

    except json.JSONDecodeError:

        start = raw_output.find("{")
        end = raw_output.rfind("}")

        if start == -1 or end == -1:

            raise ValueError(
                "No JSON object found in model response."
            )

        parsed = json.loads(
            raw_output[start:end + 1]
        )


    if "diseases" not in parsed:

        raise ValueError(
            "JSON does not contain 'diseases'."
        )


    diseases = parsed["diseases"]


    if not isinstance(diseases, list):

        raise ValueError(
            "'diseases' must be a list."
        )


    cleaned = []
    seen = set()


    for disease in diseases:

        if not isinstance(disease, str):
            continue

        disease = disease.strip()

        if not disease:
            continue

        key = disease.lower()

        if key not in seen:

            seen.add(key)
            cleaned.append(disease)


    return cleaned

In [13]:
def extract_gemma(indication_text):

    user_message = EXTRACTION_TEMPLATE.format(
        indication_text=indication_text
    )

    payload = {

        "model": MODEL,

        "messages": [

            {
                "role": "system",
                "content": SYSTEM_MESSAGE
            },

            {
                "role": "user",
                "content": user_message
            }
        ],

        "stream": False,

        "options": {

            "temperature": 0.1,

            "num_ctx": 4096,

            "num_predict": 1000
        }
    }


    start_time = time.time()


    response = requests.post(
        URL,
        json=payload,
        timeout=240
    )

    response.raise_for_status()


    raw = (
        response.json()
        ["message"]
        ["content"]
    )


    runtime = round(
        time.time() - start_time,
        2
    )


    # Keep raw response even if JSON parsing fails
    try:

        diseases = parse_output(raw)

        status = (
            "OK"
            if diseases
            else "OK_EMPTY"
        )

    except Exception:

        diseases = []
        status = "FORMAT_FAILED"


    return diseases, raw, runtime, status

In [14]:
test_text = """
Metoprolol is indicated for the treatment of hypertension and angina pectoris.
"""

diseases, raw, runtime, status = extract_gemma(
    test_text
)

print("RAW:")
print(raw)

print()

print("PARSED:")
print(diseases)

print()

print("STATUS:")
print(status)

print()

print("Runtime:", runtime)

RAW:
```json
{"diseases":["hypertension","angina pectoris"]}
```

PARSED:
['hypertension', 'angina pectoris']

STATUS:
OK

Runtime: 97.72


In [15]:
for i in range(5):

    record_id = df.loc[i, "record_id"]

    indication_text = str(
        df.loc[i, "indication_text"]
    )


    print()
    print("=" * 70)
    print(f"RECORD {i + 1}/5")
    print("Record ID:", record_id)

    print()
    print("EXPERT GOLD:")
    print(
        df.loc[i, "gold_expert_mentions"]
    )


    try:

        diseases, raw, runtime, status = extract_gemma(
            indication_text
        )


        df.at[
            i,
            "gemma_output"
        ] = json.dumps(
            diseases,
            ensure_ascii=False
        )


        df.at[
            i,
            "gemma_raw"
        ] = raw


        df.at[
            i,
            "gemma_status"
        ] = status


        df.at[
            i,
            "gemma_runtime_sec"
        ] = runtime


        print()
        print("GEMMA3 OUTPUT:")
        print(diseases)

        print(
            "Status:",
            status
        )

        print(
            "Runtime:",
            runtime,
            "seconds"
        )


    except Exception as e:

        df.at[
            i,
            "gemma_status"
        ] = "FAILED"

        df.at[
            i,
            "gemma_raw"
        ] = str(e)

        print(
            "TECHNICAL ERROR:",
            e
        )


df.to_excel(
    RESULT_FILE,
    index=False
)

print()
print("Saved:")
print(RESULT_FILE)


RECORD 1/5
Record ID: 10004

EXPERT GOLD:
Erosive Esophagitis; Gastroesophageal Reflux Disease; GERD; Zollinger Ellison Syndrome

GEMMA3 OUTPUT:
['Erosive Esophagitis', 'Gastroesophageal Reflux Disease', 'GERD', 'Zollinger Ellison Syndrome']
Status: OK
Runtime: 0.99 seconds

RECORD 2/5
Record ID: 10008

EXPERT GOLD:
Anxiety Disorders; anxiety disorder; Anxiety; Anxiety associated with depression; Panic Disorder; agoraphobia

GEMMA3 OUTPUT:
['Anxiety Disorders', 'Generalized anxiety disorder', 'Anxiety', 'Panic Disorder', 'Agoraphobia', 'Depression']
Status: OK
Runtime: 1.16 seconds

RECORD 3/5
Record ID: 10012

EXPERT GOLD:
ventricular fibrillation; ventricular tachycardia; VT; VF

GEMMA3 OUTPUT:
['ventricular fibrillation', 'ventricular tachycardia']
Status: OK
Runtime: 0.78 seconds

RECORD 4/5
Record ID: 10016

EXPERT GOLD:
dermatoses; plaque type psoriasis

GEMMA3 OUTPUT:
['corticosteroid responsive dermatoses', 'plaque type psoriasis', 'adrenal suppression']
Status: OK
Runtime: 0.

In [16]:
display(
    df[
        [
            "record_id",
            "gold_expert_mentions",
            "gemma_output",
            "gemma_status",
            "gemma_runtime_sec"
        ]
    ].head(5)
)

,record_id,gold_expert_mentions,gemma_output,gemma_status,gemma_runtime_sec
0,10004,Erosive Esophagitis; Gastroesophageal Reflux D...,"[""Erosive Esophagitis"", ""Gastroesophageal Refl...",OK,0.99
1,10008,Anxiety Disorders; anxiety disorder; Anxiety; ...,"[""Anxiety Disorders"", ""Generalized anxiety dis...",OK,1.16
2,10012,ventricular fibrillation; ventricular tachycar...,"[""ventricular fibrillation"", ""ventricular tach...",OK,0.78
3,10016,dermatoses; plaque type psoriasis,"[""corticosteroid responsive dermatoses"", ""plaq...",OK,0.68
4,10017,"acute, painful","[""acute, painful musculoskeletal conditions""]",OK,0.46


In [18]:
def run_all_gemma():

    total = len(df)


    for i in range(total):

        record_id = df.loc[
            i,
            "record_id"
        ]


        current_status = str(
            df.at[
                i,
                "gemma_status"
            ]
        ).strip()


        # Skip completed experimental results
        if current_status in [
            "OK",
            "OK_EMPTY",
            "FORMAT_FAILED"
        ]:

            print(
                f"[{i + 1}/{total}] "
                f"Record {record_id} - already completed"
            )

            continue


        indication_text = str(
            df.loc[
                i,
                "indication_text"
            ]
        )


        print()
        print("=" * 70)

        print(
            f"[{i + 1}/{total}] "
            f"Record ID: {record_id}"
        )


        try:

            diseases, raw, runtime, status = extract_gemma(
                indication_text
            )


            df.at[
                i,
                "gemma_output"
            ] = json.dumps(
                diseases,
                ensure_ascii=False
            )


            df.at[
                i,
                "gemma_raw"
            ] = raw


            df.at[
                i,
                "gemma_status"
            ] = status


            df.at[
                i,
                "gemma_runtime_sec"
            ] = runtime


            print(
                "Output:",
                diseases
            )

            print(
                "Status:",
                status
            )

            print(
                "Runtime:",
                runtime,
                "seconds"
            )


        except Exception as e:

            df.at[
                i,
                "gemma_status"
            ] = "FAILED"

            df.at[
                i,
                "gemma_raw"
            ] = str(e)


            print(
                "TECHNICAL ERROR:",
                e
            )


        # Checkpoint every 20 records
        if (
            (i + 1) % 20 == 0
            or
            i == total - 1
        ):

            df.to_excel(
                RESULT_FILE,
                index=False
            )

            print(
                ">>> CHECKPOINT SAVED TO GOOGLE DRIVE"
            )


    # Final save
    df.to_excel(
        RESULT_FILE,
        index=False
    )


    print()
    print("=" * 70)
    print("GEMMA3 FINISHED")
    print("=" * 70)

    print(
        "Results saved to:"
    )

    print(
        RESULT_FILE
    )

In [19]:
run_all_gemma()

[1/500] Record 10004 - already completed
[2/500] Record 10008 - already completed
[3/500] Record 10012 - already completed
[4/500] Record 10016 - already completed
[5/500] Record 10017 - already completed

[6/500] Record ID: 1001
Output: ['Hypertension', 'Angina Pectoris', 'Myocardial Infarction']
Status: OK
Runtime: 0.77 seconds

[7/500] Record ID: 10140
Output: ['major depressive disorder', 'generalized seizures', 'Major Depression']
Status: OK
Runtime: 0.69 seconds

[8/500] Record ID: 10146
Output: ['insomnia']
Status: OK
Runtime: 0.53 seconds

[9/500] Record ID: 10153
Output: ['anxiety', 'tension', 'psychoneurosis', 'organic disease states', 'pruritus', 'allergic conditions', 'chronic urticaria', 'atopic dermatoses', 'contact dermatoses', 'histamine mediated pruritus']
Status: OK
Runtime: 1.27 seconds

[10/500] Record ID: 10157
Output: ["Parkinson's disease", "idiopathic Parkinson's disease"]
Status: OK
Runtime: 0.75 seconds

[11/500] Record ID: 10173
Output: ['partial onset seizur

In [20]:
print(
    df["gemma_status"]
    .value_counts(dropna=False)
)

gemma_status
OK               497
FORMAT_FAILED      3
Name: count, dtype: int64


In [21]:
failed_gemma = df[
    df["gemma_status"] == "FORMAT_FAILED"
]

display(
    failed_gemma[
        [
            "record_id",
            "gold_expert_mentions",
            "gemma_raw"
        ]
    ]
)

,record_id,gold_expert_mentions,gemma_raw
45,10029,"Infections of the ear, nose, and throat; Infec...","```json\n{""diseases"":[""ear infections"",""nose i..."
225,1432,Pharyngitis; Tonsillitis; Acute Bacterial Otit...,"```json\n{""diseases"":[""Pharyngitis Tonsillitis..."
258,12261,asthma; atopic dermatitis; contact dermatitis;...,"```json\n{""diseases"":[""allergic states"",""asthm..."


In [22]:
for i, row in failed_gemma.iterrows():

    print("=" * 80)
    print("RECORD ID:", row["record_id"])

    print()
    print("RAW RESPONSE:")
    print(repr(row["gemma_raw"]))

    print()

RECORD ID: 10029

RAW RESPONSE:
'```json\n{"diseases":["ear infections","nose infections","throat infections","pneumonia","Staphylococcus infections","influenza","genitourinary tract infections","coli infections","P. mirabilis infections","faecalis infections","skin infections","skin structure infections","lower respiratory tract infections","Streptococcus infections","hemolytic strains","gonorrhea","acute uncomplicated ano genital infections","urethral infections","duodenal ulcer disease","pylori infection"]}:}\n```'

RECORD ID: 1432

RAW RESPONSE:
'```json\n{"diseases":["Pharyngitis Tonsillitis","Acute Bacterial Otitis Media","Acute Bacterial Maxillary Sinusitis","Acute Bacterial Exacerbations of Chronic Bronchitis and Secondary Bacterial Infections of Acute Bronchitis","Uncomplicated Skin and Skin Structure Infections","Uncomplicated Urinary Tract Infections","Uncomplicated Gonorrhea","Early Lyme Disease (erythema migrans)"]}:}\n```'

RECORD ID: 12261

RAW RESPONSE:
'```json\n{"dise

In [23]:
!pip -q install json-repair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 2.9 MB/s eta 0:00:00


In [24]:
import json
from json_repair import repair_json


failed_indices = df.index[
    df["gemma_status"] == "FORMAT_FAILED"
].tolist()


for i in failed_indices:

    raw = str(
        df.at[i, "gemma_raw"]
    ).strip()

    print()
    print("=" * 70)
    print("Record:", df.at[i, "record_id"])

    try:

        # Remove markdown fences
        cleaned_raw = (
            raw
            .replace("```json", "")
            .replace("```", "")
            .strip()
        )

        # Repair JSON syntax only
        repaired = repair_json(
            cleaned_raw
        )

        parsed = json.loads(
            repaired
        )

        diseases = parsed["diseases"]


        # Remove exact duplicate strings only
        cleaned_diseases = []
        seen = set()

        for disease in diseases:

            disease = str(disease).strip()

            if (
                disease
                and disease.lower() not in seen
            ):

                seen.add(
                    disease.lower()
                )

                cleaned_diseases.append(
                    disease
                )


        df.at[
            i,
            "gemma_output"
        ] = json.dumps(
            cleaned_diseases,
            ensure_ascii=False
        )


        df.at[
            i,
            "gemma_status"
        ] = "OK_REPAIRED"


        print(
            "Recovered:",
            cleaned_diseases
        )


    except Exception as e:

        print(
            "Could not repair:",
            e
        )


df.to_excel(
    RESULT_FILE,
    index=False
)

print()
print("Saved:", RESULT_FILE)


Record: 10029
Recovered: ['ear infections', 'nose infections', 'throat infections', 'pneumonia', 'Staphylococcus infections', 'influenza', 'genitourinary tract infections', 'coli infections', 'P. mirabilis infections', 'faecalis infections', 'skin infections', 'skin structure infections', 'lower respiratory tract infections', 'Streptococcus infections', 'hemolytic strains', 'gonorrhea', 'acute uncomplicated ano genital infections', 'urethral infections', 'duodenal ulcer disease', 'pylori infection']

Record: 1432
Recovered: ['Pharyngitis Tonsillitis', 'Acute Bacterial Otitis Media', 'Acute Bacterial Maxillary Sinusitis', 'Acute Bacterial Exacerbations of Chronic Bronchitis and Secondary Bacterial Infections of Acute Bronchitis', 'Uncomplicated Skin and Skin Structure Infections', 'Uncomplicated Urinary Tract Infections', 'Uncomplicated Gonorrhea', 'Early Lyme Disease (erythema migrans)']

Record: 12261
Recovered: ['allergic states', 'asthma', 'atopic dermatitis', 'contact dermatitis',

In [25]:
print(
    df["gemma_status"]
    .value_counts(dropna=False)
)

gemma_status
OK             497
OK_REPAIRED      3
Name: count, dtype: int64
